# Module 06: Scores (configs, targets, feedback and corrections)

**Goal:** every request gets **quality numbers** that everyone defines the same way. These are:
- code scores the agent can't argue with,
- an end-user feedback hook with corrections,
- all of it attached to the right **target**, and validated before it's sent.

| Concept | What it is | In our support agent |
|---|---|---|
| **Score** | name + value + data type + **target**, with an optional comment and metadata | `handoff_correct=False` on one trace |
| **Target** | what the score is about: a **trace**, an **observation** or a **session** | trace: the answer · observation: the `kb-search` retriever · session: the whole chat |
| **Score config** | the schema for one score name: type, range, categories | `kb_hit` is NUMERIC 0..1; `user_feedback` is `thumbs_up`/`thumbs_down` |
| **Data types** | NUMERIC, BOOLEAN, CATEGORICAL (+ TEXT; and CORRECTION, which takes no config) | CORRECTION holds the *right* answer |
| **Source** | who produced it: `API` (code, users), `EVAL` (LLM judges, module 07), `ANNOTATION` (humans, module 08) | all `API` here |

```
Session  …-s_refund_followup  ─── csat = 4                      (session score, end-of-chat survey)

Trace  support-request "I cannot log in…"  ─── handoff_correct = False   (code score, trace)
  │                                        ─── user_feedback  = thumbs_down "wrong team"
  │                                        ─── output (CORRECTION) = "…routed to account security…"
  └── … tools → search_kb → kb-search (retriever) ─── kb_hit = 1.0   (code score, observation)
```

**Why configs?** Without them, team A sends `helpfulness` as 1-5, team B as 0-1, and the dashboard averages both.
A config fixes the type and range once. The server then **rejects** scores that break it, and (as the demo shows)
it rejects them *silently*.

Docs: https://langfuse.com/docs/evaluation/evaluation-methods/data-model ·
https://langfuse.com/docs/evaluation/evaluation-methods/custom-scores ·
https://langfuse.com/docs/evaluation/evaluation-methods/scores-via-ui ·
https://langfuse.com/docs/observability/features/user-feedback


In [1]:
import sys
from pathlib import Path
LAB_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "langfuse_lab").is_dir())
sys.path.insert(0, str(LAB_ROOT))  # make `langfuse_lab` / `langfuse_mastery` importable whatever the kernel's cwd

import importlib
import subprocess
import time
import uuid
from langfuse_lab import verify, traffic, kb
from langfuse_lab.config import Settings
from langfuse_lab.agent import SupportAgent
from langfuse_lab.failures import FailureConfig
from langfuse_lab.fixtures import load_fixtures
from langfuse_lab.guardrail import check_input
from langfuse_mastery import scoring

settings = Settings.from_env()  # loads the repo-root .env (Langfuse keys + Azure settings)
RUN_TAG = f"run:m06-{uuid.uuid4().hex[:6]}"
RUN_ID = RUN_TAG.split(":")[1]
ENV = "dev"
fixtures = load_fixtures()
CASES = traffic.score_demo_cases(run_id=RUN_ID)
SESSION = traffic.score_demo_session(run_id=RUN_ID)
print("run tag:", RUN_TAG)
for c in CASES:
    print(f"  {c.case_id:<36} expect handoff={c.expected_handoff:<19} kb={c.expected_kb or '-':<11} {c.failures or ''}")

run tag: run:m06-549a27
  refund_double_charge                 expect handoff=billing_specialist  kb=-           
  shipping_status_no_handoff_needed    expect handoff=none                kb=-           
  kb_refund_policy                     expect handoff=none                kb=kb_refunds  
  reset_password_wrong_handoff         expect handoff=account_security    kb=-           ('wrong_handoff',)


## Given: the instrumentation you built in modules 01–02
This is a recap, not a TODO. One addition: the retriever records `(its observation id, the article ids it returned)`
in `RETRIEVALS`. Scoring an **observation** needs its id, and the only place that knows it is the code running
inside that span (`get_current_observation_id()`).


In [2]:
from langfuse import get_client, observe, propagate_attributes
from langfuse.langchain import CallbackHandler

langfuse = get_client()
RETRIEVALS = []  # [(observation_id, [article ids])] for the turn in progress


@observe(name="guardrail", as_type="guardrail")
def observed_guardrail(text):
    return check_input(text)


@observe(name="kb-search", as_type="retriever")
def observed_search(query, k):
    results = kb.search(query, fixtures.kb, k=k)
    RETRIEVALS.append((langfuse.get_current_observation_id(), [a["id"] for a in results]))
    return results


@observe(name="support-request", as_type="agent", capture_input=False, capture_output=False)
def traced_turn(message, agent, *, session_id, user_id, tags, history=None):
    """One turn = one trace. Returns (AgentRun, trace_id) so the caller can score the trace afterwards."""
    with propagate_attributes(session_id=session_id, user_id=user_id, tags=[RUN_TAG, *tags], environment=ENV):
        run = agent.invoke(message, history=history or [], config={"callbacks": [CallbackHandler()]})
        langfuse.update_current_span(input=message, output=run.summary["final_answer"])
        return run, langfuse.get_current_trace_id()


clean_agent = SupportAgent(guardrail_fn=observed_guardrail, search_fn=observed_search)
planted_agent = SupportAgent(failures=FailureConfig.from_names(["wrong_handoff"]),  # bad prompt: login -> billing
                             guardrail_fn=observed_guardrail, search_fn=observed_search)

## TODO 1: score configs, defined once

Write `ensure_score_configs() -> dict[str, dict]`. It should get-or-create these configs **by name**:

| name | data type | range / categories | produced by |
|---|---|---|---|
| `handoff_correct` | BOOLEAN | n/a | your code (TODO 3) |
| `kb_hit` | NUMERIC | 0..1 | your code, on the retriever observation (TODO 3) |
| `user_feedback` | CATEGORICAL | `thumbs_up`=1, `thumbs_down`=0 | the end user (TODO 4) |
| `csat` | NUMERIC | 1..5 | the end user, on the session (TODO 4) |

Return `{name: config_dict}`, where `config_dict = config.dict()`. That gives camelCase keys (`id`, `dataType`,
`minValue`, `maxValue`, `categories`), the shape `scoring.validate_score` expects.

Two things the server does **not** do for you (both verified live):
- **Names are not unique.** Calling `create` twice makes two `kb_hit` configs, and your scores split between them.
  So list first, and create only what's missing.
- **Archived configs are still listed** (`is_archived=True`). Skip them.

Hints:
- `langfuse.api.score_configs.get(page=1, limit=100)` (`.data`, `.meta.total_pages`)
- `langfuse.api.score_configs.create(name=, data_type=, min_value=, max_value=, categories=[{"label": …, "value": …}],
  description=)`

The cell after yours runs it **twice** and checks that nothing new was created.


In [3]:
# TODO 1: your code here
SCORE_CONFIG_SPEC ={
    "handoff_correct" : dict(data_type='BOOLEAN',description="Agent handed off to the expected team or not"),
    "kb_hit":dict(data_type="NUMERIC",min_value=0,max_value=1,description ="Expected KB article retrived (code)" ),
    "user_feedback":dict(data_type="CATEGORICAL",description="Agent handed off to the expected team or not",
    categories=[{"label": "thumbs_up", "value": 1}, {"label": "thumbs_down", "value": 0}]),
    "csat":dict(data_type="NUMERIC",min_value=1,max_value=5,description="End-of-chat survey (session)")
}
def ensure_score_configs() -> dict[str,dict] :
    existing,page = {},1
    while True:

        resp = langfuse.api.score_configs.get(page=page,limit=100)
        for c in resp.data :
            if c.is_archived:
                continue
            existing.setdefault(c.name,c)
        if page >= int(resp.meta.total_pages or 1):
            break
        page+=1

    out = {}
    for name, spec in SCORE_CONFIG_SPEC.items():
        cfg = existing.get(name)
        if cfg is None:
            cfg=langfuse.api.score_configs.create(name=name,**spec)
        out[name]=cfg.dict()
    return out





In [4]:
resp = langfuse.api.score_configs.get(page=1, limit=100)
existing = {}
for c in resp.data:
    if c.is_archived:
        continue                       # dead config: pretend it isn't there
    existing.setdefault(c.name, c)     # first one wins if a name appears twice

print(sorted(existing))

['csat', 'handoff_correct', 'kb_hit', 'user_feedback']


In [5]:
first, second = ensure_score_configs(), ensure_score_configs()
assert {k: v["id"] for k, v in first.items()} == {k: v["id"] for k, v in second.items()}, "second call created new configs"
configs = second
for name, cfg in configs.items():
    print(f"{name:<16} {cfg['dataType']:<12} {cfg.get('minValue')}..{cfg.get('maxValue')}  {cfg['id']}")

handoff_correct  BOOLEAN      None..None  f11fef23-1596-4733-8470-0807d9ec4231
kb_hit           NUMERIC      0.0..1.0  13da9e9b-f786-4581-a90f-91cb1ebae324
user_feedback    CATEGORICAL  None..None  002f5a4e-1f26-411b-b75c-260a13c670d1
csat             NUMERIC      1.0..5.0  659f2be2-4aa1-403e-8ada-00fbd48dd4e0


## Demo: the server drops invalid scores *silently*
This cell sends two scores against your `kb_hit` config (0..1): a valid `0.5` and an invalid `7`. Then it reads
back what was stored. The SDK reports nothing, and only one score arrives.
The only evidence is a line in the **worker's** log:
`docker logs ch09_langfuse_worker --since 5m | findstr "not valid against provided config"`.

That's why TODO 2 validates scores **before** sending them.


In [6]:
original_add = langfuse._resources.add_score_task

def spy(score, **kw):
    print("QUEUED ->", score)
    return original_add(score, **kw)

langfuse._resources.add_score_task = spy

with langfuse.start_as_current_observation(name="m06-demo", as_type="span") as demo:
    demo.update(input="silent rejection demo", output="ok")
    demo_trace = langfuse.get_current_trace_id()   # not tagged RUN_TAG on purpose: verify ignores it
for value in (0.5, 7):
    langfuse.create_score(trace_id=demo_trace, name="kb_hit", value=value, data_type="NUMERIC",
                          config_id=configs["kb_hit"]["id"], comment=f"demo value {value}")
langfuse.flush()

stored = []
for _ in range(10):
    stored = langfuse.api.scores.get_many(trace_id=demo_trace, fields="details").data
    if stored:
        break
    time.sleep(3)
time.sleep(3)  # give the invalid one the same chance to arrive
stored = langfuse.api.scores.get_many(trace_id=demo_trace, fields="details").data
print("sent 2 scores, stored:", [(s.value, s.comment) for s in stored])

QUEUED -> {'id': '96461b51ce940d89', 'traceId': '8ac3164d02418df59300c8e1d790ff27', 'name': 'kb_hit', 'value': 0.5, 'comment': 'demo value 0.5', 'dataType': 'NUMERIC', 'configId': '13da9e9b-f786-4581-a90f-91cb1ebae324', 'timestamp': datetime.datetime(2026, 11, 12, 22, 48, 52, 107224, tzinfo=datetime.timezone.utc)}
QUEUED -> {'id': '9ac4f8fad6ce56f7', 'traceId': '8ac3164d02418df59300c8e1d790ff27', 'name': 'kb_hit', 'value': 7.0, 'comment': 'demo value 7', 'dataType': 'NUMERIC', 'configId': '13da9e9b-f786-4581-a90f-91cb1ebae324', 'timestamp': datetime.datetime(2026, 11, 12, 22, 48, 52, 107652, tzinfo=datetime.timezone.utc)}
sent 2 scores, stored: [(0.5, 'demo value 0.5')]


In [7]:
import json, urllib.request
from langfuse_lab import s3

score_id = "913daa19cff0a896"   # the value-7 score from your QUEUED line
st = s3.minio_settings()        # MinIO address + credentials from your .env
keys = [o["key"] for o in s3.list_objects(f"events/ch09-eval-project/score/{score_id}/")]
print(keys)

url = f"{st.endpoint}/{st.bucket}/{keys[0]}"
req = urllib.request.Request(url, headers=s3.signed_headers("GET", url, st.access_key, st.secret_key, region=st.region))
print(json.dumps(json.loads(urllib.request.urlopen(req).read()), indent=2))

['events/ch09-eval-project/score/913daa19cff0a896/963288df-d07d-458a-b674-f94043b0d309.json']
[
  {
    "id": "963288df-d07d-458a-b674-f94043b0d309",
    "timestamp": "2026-11-12T08:42:24.974049Z",
    "type": "score-create",
    "body": {
      "id": "913daa19cff0a896",
      "name": "kb_hit",
      "traceId": "05e7976cc079a1b42d5b5eae3e74a89c",
      "environment": "default",
      "comment": "demo value 7",
      "source": "API",
      "value": 7,
      "dataType": "NUMERIC",
      "configId": "13da9e9b-f786-4581-a90f-91cb1ebae324"
    }
  }
]


## TODO 2: implement `langfuse_mastery/scoring.py` (pure functions + pytest)

Open `ch09/langfuse_mastery/scoring.py` and implement:
- `validate_score(config, value)`: would Langfuse accept this value? Returns `(ok, reason)`.
- `handoff_correct(actual, expected)`
- `kb_hit(returned_ids, expected_id)`

There's no Langfuse code in that file. These are plain functions, so they're testable in milliseconds and reusable
in the CI gate (module 10). The cell below runs the tests and reloads your module. Re-run it until it shows
`32 passed`.


In [8]:
result = subprocess.run([sys.executable, "-m", "pytest", "langfuse_mastery/tests/test_scoring.py", "-q", "-p", "no:cacheprovider"],
                        cwd=LAB_ROOT, capture_output=True, text=True)
print(result.stdout[-2500:] or result.stderr[-2500:])
scoring = importlib.reload(scoring)

................................                                         [100%]Running teardown with pytest sessionfinish...

32 passed in 0.06s



## TODO 3: code scores on the trace and on the retriever observation

Write two functions.

**`send_score(configs, name, value, *, comment=None, evaluator="code:v1", score_id=None, **target)`** is the single
path every config-backed score goes through:
- It runs `scoring.validate_score(configs[name], value)` first and **raises `ValueError(reason)`** if the value is
  invalid. Fail loudly instead of the server dropping it silently.
- It calls `langfuse.create_score(name=…, value=…, data_type=…, config_id=…, comment=…, score_id=…,
  metadata={"evaluator": evaluator}, **target)`. Here `target` is `trace_id=`, `observation_id=` and/or `session_id=`.
- Send BOOLEAN values as `1.0` / `0.0`.

**`score_case(case, run, trace_id, retrievals, configs) -> bool`:**
- Trace-level `handoff_correct`: compare `run.summary["handoff_target"]` with `case.expected_handoff`. When wrong,
  comment `"expected X, got Y"` (the comment is what a reviewer reads first).
- Observation-level `kb_hit`: for each `(observation_id, ids)` in `retrievals`, if
  `scoring.kb_hit(ids, case.expected_kb)` is not None, score that observation (`trace_id=` **and**
  `observation_id=`).
- Return whether the handoff was correct.

`metadata={"evaluator": "code:v1"}` is **provenance**. When you change the scoring logic, bump the version, and you
can still tell old numbers from new ones.


In [9]:
# TODO 3: your code here
def send_score(configs, name, value, *, comment=None, evaluator="code:v1", score_id=None, **target):
    cfg = configs[name]
    ok, reason = scoring.validate_score(cfg, value)
    if not ok:
        raise ValueError(reason)
    if cfg["dataType"] == "BOOLEAN":
        value = 1.0 if value else 0.0
    langfuse.create_score(name=name, value=value, data_type=cfg["dataType"], config_id=cfg["id"],
                          comment=comment, score_id=score_id, metadata={"evaluator": evaluator}, **target)


def score_case(case, run, trace_id, retrievals, configs):
    # 1. trace-level: did the agent hand off to the right team?
    actual = run.summary["handoff_target"]
    ok = scoring.handoff_correct(actual, case.expected_handoff)
    send_score(configs, "handoff_correct", ok, trace_id=trace_id,
               comment=None if ok else f"expected {case.expected_handoff}, got {actual}")

    # 2. observation-level: did each kb-search retrieve the expected article?
    for observation_id, ids in retrievals:
        hit = scoring.kb_hit(ids, case.expected_kb)
        if hit is not None:
            send_score(configs, "kb_hit", hit, trace_id=trace_id, observation_id=observation_id)

    return ok

## TODO 4: the end-user feedback hook (votes, corrections, CSAT)

**`record_feedback(configs, trace_id, *, thumbs_up, comment="", correction=None)`:**
- Sends `user_feedback` = `"thumbs_up"` / `"thumbs_down"` through `send_score` with `evaluator="user"`.
- Uses a **deterministic** `score_id=f"{trace_id}-user_feedback"`. If the user changes their vote, the same id
  **updates** the score instead of adding a second one (verified live). One person, one vote.
- If `correction` is given, also sends `create_score(trace_id=…, name="output", value=correction,
  data_type="CORRECTION")`. CORRECTION takes no config. This is the *right* answer, and module 09 turns it into
  dataset ground truth.

**`record_csat(configs, session_id, rating)`:** a session-level `csat` (1..5) via `send_score(…, session_id=…)`.

**`feedback_report(trace_id)`:** reads the scores back with
`langfuse.api.scores.get_many(trace_id=…, fields="details,subject")` and prints name, data type, target kind,
value and comment, one per line. This is the export path pipelines use. Without `fields`, comment, config and
target are missing.


In [10]:
# TODO 4: your code here
def record_feedback(configs, trace_id, *, thumbs_up, comment="", correction=None):
    # the vote: one per user per trace, so a fixed score_id makes a changed vote overwrite the old one
    send_score(configs, "user_feedback", "thumbs_up" if thumbs_up else "thumbs_down",
               comment=comment or None, evaluator="user",
               score_id=f"{trace_id}-user_feedback", trace_id=trace_id)
    # the correction: what the answer SHOULD have been (no config, so it doesn't go through send_score)
    if correction:
        langfuse.create_score(trace_id=trace_id, name="output", value=correction,
                              data_type="CORRECTION", score_id=f"{trace_id}-correction")


def record_csat(configs, session_id, rating):
    # end-of-chat survey: about the whole conversation, so it targets the session
    send_score(configs, "csat", rating, evaluator="user",
               score_id=f"{session_id}-csat", session_id=session_id)


def feedback_report(trace_id):
    for s in langfuse.api.scores.get_many(trace_id=trace_id, fields="details,subject").data:
        d = s.dict()
        print(f"{d['name']:<16} {d['dataType']:<12} {d['subject']['kind']:<12} {d['value']!s:<40} {d.get('comment') or ''}")

In [11]:
CASES

[ScoredCase(case_id='refund_double_charge', conversation_id='m06-549a27-c000', session_id='m06-549a27-c000', user_id='cust_00', message='I was charged twice for order 8129. Please fix it.', expected_handoff='billing_specialist', expected_kb=None, reference_answer='I found the duplicate charge and handed this to billing for refund review.', failures=()),
 ScoredCase(case_id='shipping_status_no_handoff_needed', conversation_id='m06-549a27-c001', session_id='m06-549a27-c001', user_id='cust_01', message='Where is my order 9912 and when should it arrive?', expected_handoff='none', expected_kb=None, reference_answer='Order 9912 is in transit and should arrive tomorrow.', failures=()),
 ScoredCase(case_id='kb_refund_policy', conversation_id='m06-549a27-c002', session_id='m06-549a27-c002', user_id='cust_05', message='What is your refund policy?', expected_handoff='none', expected_kb='kb_refunds', reference_answer='Refunds for duplicate charges are reviewed by billing within 1 business day and 

## Run it, including the planted failure
There are four single-turn cases. The login case runs on the **`wrong_handoff`** prompt, which routes login problems
to billing. Your `handoff_correct` must catch it. The simulated user then reacts:
- right handoff: thumbs up;
- wrong handoff: thumbs **up** first (a hasty click), then changes to thumbs **down** with a comment and the
  correct answer.

After the cases, the 2-turn `s_refund_followup` chat runs and gets a CSAT of 4. That's 6 traces and about 18 Azure
calls. The cell calls your functions by these exact names.


In [ ]:
rows, planted_trace = [], None
for case in CASES:
    RETRIEVALS.clear()
    agent = planted_agent if case.failures else clean_agent
    run, trace_id = traced_turn(case.message, agent, session_id=case.session_id, user_id=case.user_id,
                                tags=[case.case_id, *case.failures])
    ok = score_case(case, run, trace_id, list(RETRIEVALS), configs)
    if ok:
        record_feedback(configs, trace_id, thumbs_up=True)
    else:
        record_feedback(configs, trace_id, thumbs_up=True)   # hasty click...
        langfuse.flush()
        time.sleep(2)                                         # ...a moment later, the user changes their mind
        record_feedback(configs, trace_id, thumbs_up=False, comment="Sent me to the wrong team.",
                        correction=case.reference_answer)
        planted_trace = trace_id
    rows.append((case.case_id, case.expected_handoff, run.summary["handoff_target"], ok, langfuse.get_trace_url(trace_id=trace_id)))

history = []
for message in SESSION.turns:
    run, _ = traced_turn(message, clean_agent, session_id=SESSION.session_id, user_id=SESSION.user_id,
                         tags=[SESSION.intent], history=history)
    history = run.messages
record_csat(configs, SESSION.session_id, 4)
langfuse.flush()

print(f"{'case':<36} {'expected':<19} {'actual':<19} handoff_correct")
for case_id, expected, actual, ok, url in rows:
    print(f"{case_id:<36} {expected:<19} {actual:<19} {ok}   {url}")

QUEUED -> {'id': 'ba5d02f6b8bb65ff', 'traceId': '2ac019a0c31089f127acc6fef1de7406', 'name': 'handoff_correct', 'value': 1.0, 'dataType': <ScoreConfigDataType.BOOLEAN: 'BOOLEAN'>, 'configId': 'f11fef23-1596-4733-8470-0807d9ec4231', 'timestamp': datetime.datetime(2026, 11, 12, 13, 23, 1, 548244, tzinfo=datetime.timezone.utc), 'metadata': {'evaluator': 'code:v1'}}
QUEUED -> {'id': '2ac019a0c31089f127acc6fef1de7406-user_feedback', 'traceId': '2ac019a0c31089f127acc6fef1de7406', 'name': 'user_feedback', 'value': 'thumbs_up', 'dataType': <ScoreConfigDataType.CATEGORICAL: 'CATEGORICAL'>, 'configId': '002f5a4e-1f26-411b-b75c-260a13c670d1', 'timestamp': datetime.datetime(2026, 11, 12, 13, 23, 1, 548429, tzinfo=datetime.timezone.utc), 'metadata': {'evaluator': 'user'}}
QUEUED -> {'id': '5b710ac26f4b6b28', 'traceId': 'c276d32feea33fe0be78d6709bf3735c', 'name': 'handoff_correct', 'value': 1.0, 'dataType': <ScoreConfigDataType.BOOLEAN: 'BOOLEAN'>, 'configId': 'f11fef23-1596-4733-8470-0807d9ec4231', 

In [ ]:
time.sleep(5)  # ingestion is async
if planted_trace:
    feedback_report(planted_trace)
else:
    print("the planted case was scored correct: check score_case (verify will flag it too)")

output           CORRECTION   trace        I checked your account and routed this to account security to restore access. 
handoff_correct  BOOLEAN      trace        False                                    expected account_security, got billing_specialist
user_feedback    CATEGORICAL  trace        thumbs_down                              Sent me to the wrong team.


`expected_traces=6` = 4 cases + 2 session turns. Verify waits for all of them, then checks:
- the configs exist exactly once,
- scores sit on the right targets and reference their configs,
- one vote per trace,
- the planted `wrong_handoff` was scored `handoff_correct=False`.


In [ ]:
verify.check_module_06(RUN_TAG, session_id=SESSION.session_id, expected_traces=6)

✅ score configs defined once: 4 configs, one each: handoff_correct, user_feedback, kb_hit, csat
✅ every case has a trace-level code score: all 4 case trace(s) have a trace-level handoff_correct score
✅ observation-level score on kb-search: kb_hit=1.0 on kb-search observation 162fdde730f72b6c (trace c3f0dcf0e25ea90a7e80872442ef7268)
✅ session-level score: csat=4 on session m06-808fcd-c004-s_refund_followup
✅ user feedback + correction (one vote per trace): 1 planted trace(s): one thumbs_down with comment + a correction
✅ scores linked to configs + provenance: 9 score(s) reference their config; code scores carry metadata.evaluator
✅ planted wrong_handoff caught by handoff_correct: 1 planted wrong_handoff trace(s) scored handoff_correct=False

MODULE PASSED


True

## UI tour
- [ ] **Traces** table: add the **Scores** columns. Filter on `handoff_correct = false`: one click to every bad
  handoff.
- [ ] Open the planted trace: the **Scores** tab shows code score, feedback (one vote, with comment) and the
  correction.
- [ ] Open the `kb-search` observation of the refund-policy trace: `kb_hit` is attached **to that span**, not to the trace.
- [ ] **Sessions** → `…-s_refund_followup`: the `csat` score sits on the session itself.
- [ ] **Settings → Scores / Score Configs**: your 4 configs (and archived probe configs, if any).
- [ ] **Scores** analytics (Dashboards / score analytics): the distribution of `handoff_correct` and `user_feedback`.

## Production notes
- **Define configs once, as code** (like `ensure_score_configs`, run at deploy time) and reference them by id. Never
  create configs from request-handling code.
- **Validate before sending.** The server drops invalid scores silently, and a mis-scaled score (0–100 into a 0–1
  config) leaves your quality dashboard empty without any error.
- **One vote per user per trace:** use a deterministic `score_id`, so a changed vote updates the score.
- **Provenance in metadata** (`evaluator: code:v1`) lets you tell numbers from old and new scoring logic apart.
- **Corrections are ground truth.** Every thumbs-down with a correction is a future regression test (module 09).
- **Browser feedback:** the frontend should call *your* backend, which writes the score. Never ship the secret key to
  a browser. Langfuse's public-key-only web SDK exists for exactly this.
- **No PII in comments**: they're shown everywhere.
- **Why it matters (leadership):** one definition of "quality", measured on every request, that every team reports
  against. Every unhappy customer leaves behind the correct answer.
